# 01 COPD Preprocessing Pipeline
Objective 1 notebook for Google Colab T4.


In [ ]:
!pip -q install librosa==0.10.2.post1 noisereduce==3.0.3 audiomentations==0.37.0 soundfile==0.12.1 scipy==1.14.1 joblib==1.4.2 scikit-learn==1.5.2 pandas==2.2.3 numpy==1.26.4 matplotlib==3.9.2 seaborn==0.13.2 tqdm==4.67.1
import torch, numpy as np, random
print('CUDA available:', torch.cuda.is_available())
SEED = 42
np.random.seed(SEED); random.seed(SEED)


In [ ]:
# Kaggle download (requires kaggle.json in Colab)
!kaggle datasets download -d vbookshelf/respiratory-sound-database -p /content --force
!unzip -o /content/respiratory-sound-database.zip -d /content/data


In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa

# Clone if needed:
# !git clone https://github.com/Rajath2005/COPD-Detection.git /content/COPD-Detection

REPO = Path('/content/COPD-Detection')
DATA_DIR = Path('/content/data')
sys.path.append(str(REPO / 'src'))
from preprocess import parse_filename_meta, parse_annotation, denoise_multistage, extract_log_mel, extract_mfcc_stack, build_augmenter, augment_audio, spec_augment, PipelineConfig

wavs = sorted(DATA_DIR.rglob('*.wav'))
diag_path = sorted(DATA_DIR.rglob('patient_diagnosis.csv'))[0]
diag = pd.read_csv(diag_path, header=None, names=['patient_id', 'diagnosis'])
diag['patient_id'] = diag['patient_id'].astype(str)
print('Total wav files:', len(wavs))
print('Diagnosis file:', diag_path)


In [ ]:
# EDA 1: Diagnosis distribution
plt.figure(figsize=(8,4))
sns.countplot(data=diag, x='diagnosis', order=diag['diagnosis'].value_counts().index)
plt.xticks(rotation=45); plt.title('Diagnosis distribution')
plt.tight_layout(); plt.show()


In [ ]:
# EDA 2: Recording metadata by diagnosis
meta = pd.DataFrame([parse_filename_meta(w.name) for w in wavs])
meta = meta.merge(diag, on='patient_id', how='left')
plt.figure(figsize=(8,4))
sns.countplot(data=meta, x='diagnosis', order=meta['diagnosis'].value_counts().index)
plt.xticks(rotation=45); plt.title('Recordings per diagnosis')
plt.tight_layout(); plt.show()


In [ ]:
# EDA 3: Duration histogram (sample for speed)
durations = []
for w in wavs[:200]:
    y, sr = librosa.load(str(w), sr=None, mono=True)
    durations.append(len(y)/sr)
plt.figure(figsize=(7,4))
plt.hist(durations, bins=30)
plt.title('Duration histogram (first 200 files)')
plt.xlabel('seconds')
plt.tight_layout(); plt.show()


In [ ]:
# EDA 4: Chest location distribution
plt.figure(figsize=(7,4))
sns.countplot(data=meta, x='chest_loc', order=meta['chest_loc'].value_counts().index)
plt.title('Chest location distribution')
plt.tight_layout(); plt.show()


In [ ]:
# EDA 5 + 6: Cycles per recording and adventitious prevalence
cycle_counts = []
crackle = wheeze = both = total = 0
for w in wavs:
    cycles = parse_annotation(w.with_suffix('.txt'))
    cycle_counts.append(len(cycles))
    for c in cycles:
        total += 1
        crackle += int(c['crackle'] == 1)
        wheeze += int(c['wheeze'] == 1)
        both += int(c['crackle'] == 1 and c['wheeze'] == 1)

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].hist(cycle_counts, bins=30)
ax[0].set_title('Cycles per recording')
ax[1].bar(['crackle','wheeze','both'], [crackle/total, wheeze/total, both/total])
ax[1].set_ylim(0,1)
ax[1].set_title('Adventitious prevalence')
plt.tight_layout(); plt.show()


In [ ]:
# Noise reduction comparison: raw vs 3-stage denoise output
example = wavs[0]
y_raw, _ = librosa.load(str(example), sr=22050, mono=True)
cfg = PipelineConfig(data_dir=Path('.'), output_dir=Path('.'))
y_dn = denoise_multistage(y_raw, 22050, cfg)

S_raw = librosa.power_to_db(librosa.feature.melspectrogram(y=y_raw, sr=22050, n_mels=128), ref=np.max)
S_dn  = librosa.power_to_db(librosa.feature.melspectrogram(y=y_dn,  sr=22050, n_mels=128), ref=np.max)
fig, ax = plt.subplots(2,2, figsize=(12,6))
ax[0,0].plot(y_raw[:5000]); ax[0,0].set_title('Raw waveform')
ax[0,1].plot(y_dn[:5000]);  ax[0,1].set_title('Denoised waveform')
ax[1,0].imshow(S_raw, aspect='auto', origin='lower'); ax[1,0].set_title('Raw Mel')
ax[1,1].imshow(S_dn,  aspect='auto', origin='lower'); ax[1,1].set_title('Denoised Mel')
plt.tight_layout(); plt.show()


In [ ]:
# Feature visualization (Log-Mel and MFCC+Δ+Δ²)
mel = extract_log_mel(y_dn, 22050)[0]
mf = extract_mfcc_stack(y_dn, 22050)
fig, ax = plt.subplots(1,4, figsize=(16,4))
ax[0].imshow(mel, aspect='auto', origin='lower'); ax[0].set_title('Log-Mel')
ax[1].imshow(mf[0], aspect='auto', origin='lower'); ax[1].set_title('MFCC')
ax[2].imshow(mf[1], aspect='auto', origin='lower'); ax[2].set_title('Delta')
ax[3].imshow(mf[2], aspect='auto', origin='lower'); ax[3].set_title('Delta2')
plt.tight_layout(); plt.show()


In [ ]:
# Augmentation showcase + SpecAugment
aug = build_augmenter(42)
augs = [augment_audio(y_dn, 22050, aug) for _ in range(4)]
fig, ax = plt.subplots(5,1, figsize=(12,7), sharex=True)
ax[0].plot(y_dn); ax[0].set_title('original')
for i, a in enumerate(augs, 1):
    ax[i].plot(a); ax[i].set_title(f'aug_{i}')
plt.tight_layout(); plt.show()

m1 = extract_log_mel(y_dn, 22050)
m2 = spec_augment(m1.copy())
fig, ax = plt.subplots(1,2, figsize=(10,4))
ax[0].imshow(m1[0], aspect='auto', origin='lower'); ax[0].set_title('Before SpecAugment')
ax[1].imshow(m2[0], aspect='auto', origin='lower'); ax[1].set_title('After SpecAugment')
plt.tight_layout(); plt.show()


In [ ]:
# Run pipeline
!python /content/COPD-Detection/src/preprocess.py --data_dir /content/data --output_dir /content/data/features --seed 42 --n_jobs -1 --copd_aug_factor 4


In [ ]:
# Final summary + leakage checks
out = Path('/content/data/features')
meta = pd.read_csv(out / 'metadata.csv')
cfg = json.loads((out / 'config.json').read_text())
print('metadata rows:', len(meta), 'patients:', meta['patient_id'].nunique())
for s in ['train','val','test']:
    m = meta[meta['split']==s]
    print(s, 'rows', len(m), 'patients', m['patient_id'].nunique(), 'binary', m['binary'].value_counts().to_dict())
pt = {s:set(meta[meta['split']==s]['patient_id']) for s in ['train','val','test']}
print('train-val overlap', len(pt['train'] & pt['val']))
print('train-test overlap', len(pt['train'] & pt['test']))
print('val-test overlap', len(pt['val'] & pt['test']))
print('X_mel_train', np.load(out/'X_mel_train.npy').shape)
print('X_mfcc_train', np.load(out/'X_mfcc_train.npy').shape)
print('config keys:', list(cfg.keys()))


In [ ]:
# Zip and download features
import shutil
from google.colab import files
zip_path = shutil.make_archive('/content/copd_features', 'zip', '/content/data/features')
files.download(zip_path)
